# Ask CIP reader v9: retrained on labels a person checked

Ask's reader (a small MiniLM model, 23 MB, on the server's CPU) picks which survey question answers a question from the
list the search finds. It learnt one big model's picks in v6, and that model's prompt bias came with it ("a topic alone
means its main question", "prefer many waves"). v9 keeps a person in the loop:

1. **The big model proposes** a pick for every real and dev question and for an anchor pool of analysts' asks: the
   largest open model this GPU holds (Qwen2.5-72B on an A100 80 GB, Qwen2.5-32B on an A100 40 GB, Qwen2.5-14B on an L4 or a
   T4), each list read twice, in order and reversed; a pick counts only when both readings agree.
2. **You check a short sheet**: only the real and dev questions where the big model and today's reader disagree, one
   row per question stem. `reader-v9-sheet.csv` downloads and the notebook stops.
3. **Run all again** with the checked sheet: a new reader learns from your checked rows and from every question where
   the big model and today's reader agree, is shrunk to the server's 8-bit format and downloads as `ask-reader-v9.zip`.
   Send it back: it is scored inside Ask on every test set (real and dev questions it never trained on included), not
   alone.

**Run 1:** Runtime → Change runtime type → **A100** (an L4 or T4 works, with a smaller model) → Run all → pick
`reader-v9.json` from `Downloads/cip-colab`. About 30-60 minutes. Keep `teacher-v9.json` and `reader-v9-sheet.csv`.

**Check the sheet:** in the `your answer` column write **R** (today's reader is right), **M** (the model is right), the
number of another choice from the `choices` column, or **0** (none of them answers it). Leave a row blank to skip it.
Save it as `reader-v9-sheet-checked.csv`.

**Run 2:** Run all → pick `reader-v9.json`, `teacher-v9.json` and `reader-v9-sheet-checked.csv`. About 15 minutes.
No API key is used anywhere; the "HF_TOKEN" warning is harmless.

In [ ]:
# 1. Libraries (vLLM brings its own PyTorch, so it goes first), the GPU, the files
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "onnx", "onnxruntime"], check=True)
import csv, json, os, random, shutil, time, zipfile
import numpy as np, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100 (or L4, T4), then Run all again"
MEMORY = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{MEMORY:.0f} GB")
if not os.path.exists("reader-v9.json"):
    from google.colab import files
    print("Pick reader-v9.json (and, for run 2, teacher-v9.json and reader-v9-sheet-checked.csv) from Downloads/cip-colab")
    files.upload()
data = json.load(open("reader-v9.json"))
items = data["items"]
print(len(items), "questions:", {k: sum(1 for i in items if i["set"] == k) for k in ("real", "dev", "pool")})

In [ ]:
%%writefile teacher.py
# The big model, in its own process so the GPU is free when it exits: the largest that loads on this GPU proposes a pick
# for every question, each candidate list read in order and reversed (a position bias shows as two different picks).
# Saves as it goes; a run cut short goes on from where it stopped.
import json, os, re, sys, torch
from vllm import LLM, SamplingParams
items = json.load(open("reader-v9.json"))["items"]
memory = torch.cuda.get_device_properties(0).total_memory / 1e9
MODELS = [("Qwen/Qwen2.5-72B-Instruct-AWQ", 75), ("Qwen/Qwen2.5-32B-Instruct-AWQ", 38), ("Qwen/Qwen2.5-14B-Instruct-AWQ", 14)]
SYSTEM = "You match an analyst's question to the consumer survey question that answers it."
def prompt(item, order):
    lines = "\n".join(f"{n}. {item['lines'][i]}" for n, i in enumerate(order, 1))
    return (f"Analyst's question: {item['reads']}\n\nSurvey questions (waves asked, years, name: wording | some answers):\n{lines}\n\n"
            "Which survey question answers the analyst's question as it is asked? Judge only by what each one asks, not by how "
            "often or how recently it was asked. Reply with its number only, or 0 if none of them answers it.")
done = json.load(open("teacher-v9.json")) if os.path.exists("teacher-v9.json") else {}
llm = None
for name, need in MODELS:
    if memory < need:
        continue
    try:
        llm = LLM(name, max_model_len=4096, gpu_memory_utilization=0.92, enforce_eager=memory < 30)
        break
    except Exception as exc:   # too big after all (its working memory): the next one down
        print(f"{name} did not load ({type(exc).__name__}); trying a smaller one", flush=True)
        torch.cuda.empty_cache()
assert llm is not None, "No model fits this GPU"
done["model"] = name
print("Model:", name, flush=True)
params = SamplingParams(temperature=0, max_tokens=6)
todo = [i for i in range(len(items)) if str(i) not in done]
for start in range(0, len(todo), 256):
    batch = todo[start:start + 256]
    orders = [(list(range(len(items[i]["lines"]))), list(reversed(range(len(items[i]["lines"]))))) for i in batch]
    chats = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt(items[i], o)}] for i, pair in zip(batch, orders) for o in pair]
    replies = llm.chat(chats, params, use_tqdm=False)
    for k, i in enumerate(batch):
        picks = []
        for o, reply in zip(orders[k], replies[2 * k:2 * k + 2]):
            found = re.search(r"\d+", reply.outputs[0].text)
            n = int(found[0]) if found else -1
            picks.append(None if n == 0 else o[n - 1] if 1 <= n <= len(o) else -1)   # -1: an unreadable reply
        done[str(i)] = picks
    json.dump(done, open("teacher-v9.json", "w"))
    print(f"{min(start + 256, len(todo))} of {len(todo)}", flush=True)

In [ ]:
# 2. The big model's proposals (skipped when teacher-v9.json is complete), then downloaded to keep
done = json.load(open("teacher-v9.json")) if os.path.exists("teacher-v9.json") else {}
if any(str(i) not in done for i in range(len(items))):
    run = subprocess.Popen([sys.executable, "teacher.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in run.stdout:   # its progress, without vLLM's own chatter
        if not line.startswith(("INFO", "WARNING", "(", "[")) and line.strip():
            print(line.rstrip())
    assert run.wait() == 0, "The big model stopped: see above"
    done = json.load(open("teacher-v9.json"))
    from google.colab import files
    files.download("teacher-v9.json")
UNSURE = "unsure"
def proposal(i):   # the pick both readings agree on (None: none of them), else unsure
    a, b = done[str(i)]
    return a if a == b and a != -1 else UNSURE
proposals = [proposal(i) for i in range(len(items))]
agree = sum(p != UNSURE and p == it["today"] for p, it in zip(proposals, items))
print(f"{done['model']}: sure on {sum(p != UNSURE for p in proposals)} of {len(items)}; agrees with today's reader on {agree}")

In [ ]:
# 3. The sheet: only real and dev questions (the stems trained on) where the big model and today's reader disagree, or
# the big model is unsure, one row per stem and pair of picks. Run 2 reads your answers from the checked copy.
def text(it, pick):
    return "none of them" if pick is None else "(unsure)" if pick == UNSURE else it["texts"][pick][:160]
rows = {}
for i, (it, p) in enumerate(zip(items, proposals)):
    if it["set"] == "pool" or it["split"] != "train" or p == it["today"]:
        continue
    key = (it["stem"], text(it, it["today"]), text(it, p))
    rows.setdefault(key, {"items": [], "it": it, "p": p})["items"].append(i)
sheet = []
for n, ((stem, today, model), r) in enumerate(sorted(rows.items()), 1):
    it = r["it"]
    sheet.append({"row": n, "question": it["q"], "also covers": len(r["items"]) - 1, "today's reader": today, "the model": model,
                  "test file expects": "none" if it["want"] == "none" else "; ".join(it["want"])[:200],
                  "choices": "\n".join(f"{k + 1}. {t[:140]}" for k, t in enumerate(it["texts"])), "your answer": "",
                  "_items": r["items"]})
fields = ["row", "question", "also covers", "today's reader", "the model", "test file expects", "choices", "your answer"]
with open("reader-v9-sheet.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fields, extrasaction="ignore"); w.writeheader(); w.writerows(sheet)
print(f"{len(sheet)} rows to check (covering {sum(len(s['_items']) for s in sheet)} questions)")
if not os.path.exists("reader-v9-sheet-checked.csv"):
    from google.colab import files
    files.download("reader-v9-sheet.csv")
assert os.path.exists("reader-v9-sheet-checked.csv"), (
    "Check reader-v9-sheet.csv (R, M, a choice's number or 0 in 'your answer'), save it as reader-v9-sheet-checked.csv, "
    "then Run all again and pick it with reader-v9.json and teacher-v9.json. This stop is on purpose.")

In [ ]:
# 4. The labels: every checked row, three times (they are few and a person read them), and every question where the big
# model and today's reader agree (two independent readings: v6 learnt the big model alone and copied its bias)
checked = {int(r["row"]): r["your answer"].strip().upper() for r in csv.DictReader(open("reader-v9-sheet-checked.csv"))}
labels, used = {}, {"checked": 0, "skipped": 0}
for s in sheet:
    answer = checked.get(s["row"], "")
    it = items[s["_items"][0]]
    if answer == "R":
        pick = it["today"]
    elif answer == "M" and s["the model"] != "(unsure)":
        pick = proposals[s["_items"][0]]
    elif answer.isdigit() and int(answer) <= len(it["texts"]):
        pick = None if answer == "0" else int(answer) - 1
    else:
        used["skipped"] += 1
        continue
    for i in s["_items"]:   # the same stem's other questions: the same pick when they read the same list
        if items[i]["texts"] == it["texts"]:
            labels[i] = ("checked", pick)
    used["checked"] += 1
for i, (it, p) in enumerate(zip(items, proposals)):
    if it["split"] == "train" and i not in labels and p != UNSURE and p == it["today"]:
        labels[i] = ("agreed", p)
train_ex = [(items[i]["reads"], items[i]["lines"], p) for i, (how, p) in labels.items() for _ in range(3 if how == "checked" else 1)]
held = [(it["reads"], it["lines"], p) for it, p in zip(items, proposals) if it["set"] == "pool" and it["split"] == "held"
        and p != UNSURE and p == it["today"]]
checked_ex = [(items[i]["reads"], items[i]["lines"], p) for i, (how, p) in labels.items() if how == "checked"]
print(f"rows checked {used['checked']}, skipped {used['skipped']} · trained on {len(train_ex)} (checked x3) · "
      f"watched on {len(held)} pool questions it never sees")

In [ ]:
# 5. The new reader: a score per candidate line; "none of these" a fixed 0, so a candidate is picked only above 0 (as v6
# and api/ask_model.py read_pick). The same base and size as today's reader: the server's speed and memory do not change
from transformers import AutoModelForSequenceClassification, AutoTokenizer
DEVICE = "cuda"
BASE = "cross-encoder/ms-marco-MiniLM-L6-v2"
def scores(model, tok, question, lines):
    enc = tok([question] * len(lines), lines, truncation=True, max_length=192, padding=True, return_tensors="pt").to(DEVICE)
    return model(**enc).logits.view(-1)
def choose(s):
    best = int(torch.argmax(s))
    return best if float(s[best]) > 0 else None
def agreement(model, tok, rows):
    model.eval()
    with torch.no_grad():
        return float(np.mean([choose(scores(model, tok, q, l)) == p for q, l, p in rows])) if rows else float("nan")
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1).to(DEVICE)
EPOCHS, PER_STEP, LR = 4, 16, 3e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * (len(train_ex) // PER_STEP)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
best, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train(); random.Random(epoch).shuffle(train_ex); started = time.time()
    for s in range(0, len(train_ex) - PER_STEP + 1, PER_STEP):
        loss = 0
        for q, lines, p in train_ex[s:s + PER_STEP]:   # softmax over "none" (0) and every candidate
            logits = torch.cat([torch.zeros(1, device=DEVICE), scores(model, tok, q, lines)])
            loss = loss + torch.nn.functional.cross_entropy(logits.unsqueeze(0), torch.tensor([0 if p is None else p + 1], device=DEVICE))
        (loss / PER_STEP).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
    now = agreement(model, tok, held)
    print(f"epoch {epoch + 1}: right on {now:.1%} of the watched pool, {agreement(model, tok, checked_ex):.1%} of the checked rows · {time.time() - started:.0f}s")
    if now > best:
        best, best_state = now, {k: v.detach().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); model.eval()

In [ ]:
# 6. Shrunk to the server's format (8-bit ONNX), read as api/ask_model.py read_pick reads it (its lead in logits over
# the next choice, "none" = 0). Its "sure" lead keeps today's share of sure picks on the watched pool, so Ask's rules
# that depend on it (api/assistant.py READER_SURE) see as many sure picks as before
import onnxruntime, tokenizers
from onnxruntime.quantization import quantize_dynamic, QuantType
NAMES = ["input_ids", "attention_mask", "token_type_ids"]
FOLDER = "ask-reader-v9"
os.makedirs(FOLDER, exist_ok=True)
dummy = tok(["a question"], ["a survey question"], return_tensors="pt")
torch.onnx.export(model.cpu().eval(), tuple(dummy[k] for k in NAMES), "full.onnx", input_names=NAMES, output_names=["logits"],
                  dynamic_axes={k: {0: "batch", 1: "tokens"} for k in NAMES} | {"logits": {0: "batch"}}, opset_version=17, dynamo=False)
quantize_dynamic("full.onnx", f"{FOLDER}/model_optimized.onnx", weight_type=QuantType.QInt8)
tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{FOLDER}/tokenizer.json")
words = tokenizers.Tokenizer.from_file(f"{FOLDER}/tokenizer.json"); words.enable_truncation(192); words.enable_padding(pad_id=0, pad_token="[PAD]")
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1
session = onnxruntime.InferenceSession(f"{FOLDER}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def server_pick(question, lines):   # read_pick, line for line
    coded = words.encode_batch([(question, t) for t in lines])
    feed = {"input_ids": np.array([c.ids for c in coded], np.int64), "attention_mask": np.array([c.attention_mask for c in coded], np.int64),
            "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
    choices = np.concatenate([[0.0], session.run(None, feed)[0].reshape(-1)])
    top, second = np.argsort(-choices)[:2]
    return (int(top) - 1 if top else None), float(choices[top] - choices[second])
watched = [it for it in items if it["set"] == "pool" and it["split"] == "held"]
leads = sorted(server_pick(it["reads"], it["lines"])[1] for it in watched)
share_sure = float(np.mean([it["lead"] >= data["reader_sure"] for it in watched]))
sure = round(float(np.quantile(leads, 1 - share_sure)), 3)
same = float(np.mean([server_pick(q, l)[0] == p for q, l, p in held]))
json.dump({"sure": sure, "length": 192, "made": f"scripts/colab/train_reader_v9.ipynb ({done['model']} proposals, {used['checked']} checked rows)",
           "reads": "api/ask_model.py describe_candidate, the candidate as the AI reads it",
           "sure_means": f"lead in logits from which the pick is final: {share_sure:.0%} of the watched pool, as today's reader at {data['reader_sure']}"},
          open(f"{FOLDER}/calibration.json", "w"), indent=1)
print(f"8-bit: right on {same:.1%} of the watched pool · sure from a lead of {sure} ({share_sure:.0%} of picks, as today)")

In [ ]:
# 7. The zip: the new reader (a server models/ask-reader folder), the proposals, the checked sheet and the labels used
json.dump({"labels": {str(i): [how, p] for i, (how, p) in labels.items()}, "model": done["model"], "used": used},
          open("labels-v9.json", "w"))
with zipfile.ZipFile("ask-reader-v9.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in os.listdir(FOLDER):
        z.write(f"{FOLDER}/{name}")
    for name in ("teacher-v9.json", "reader-v9-sheet-checked.csv", "labels-v9.json"):
        z.write(name)
from google.colab import files
files.download("ask-reader-v9.zip")
print("Send ask-reader-v9.zip back: it is scored inside Ask on every test set before it replaces models/ask-reader.")